# 생산·품질 데이터 분석 — Google Drive 실행용
GitHub에서 내려받은 자료를 **내 Google Drive에 저장한 뒤** Colab으로 실행합니다.
데이터는 교육용 합성 데이터입니다.


In [ ]:
# Google Drive에 저장한 CSV 파일을 읽습니다.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path('/content/drive/MyDrive/production_quality_ai_workshop/data')
assert DATA_DIR.is_dir(), f'데이터 폴더를 찾을 수 없습니다: {DATA_DIR}'
production = pd.read_csv(DATA_DIR / 'production_daily.csv', parse_dates=['date'])
defects = pd.read_csv(DATA_DIR / 'defect_types.csv', parse_dates=['date'])
print('생산 데이터:', production.shape, '불량 유형 데이터:', defects.shape)
display(production.head())


## 1. 데이터 검증
일자·라인별 불량 유형 합계가 생산 데이터의 불량 수량과 일치하는지 확인합니다.

In [ ]:
assert not production.duplicated(["date", "line"]).any(), "일자·라인 중복"
assert production[["planned_units","produced_units","defect_units","scheduled_hours","downtime_hours"]].notna().all().all()
assert (production["defect_units"] <= production["produced_units"]).all()
assert (production["produced_units"] > 0).all()
assert (production["scheduled_hours"] > production["downtime_hours"]).all()
check = defects.groupby(["date","line"],as_index=False)["count"].sum().merge(
    production[["date","line","defect_units"]],on=["date","line"],how="outer",indicator=True)
assert (check["_merge"] == "both").all() and (check["count"] == check["defect_units"]).all()
print("검증 통과: 중복/누락/불량 수량 일치")


## 2. KPI 계산
비율의 집계는 **합계 ÷ 합계**를 사용합니다. 시간가동률은 OEE가 아닙니다.

In [ ]:
production["operating_hours"] = production["scheduled_hours"] - production["downtime_hours"]
summary = production.groupby("line",as_index=False).agg(
    planned_units=("planned_units","sum"), produced_units=("produced_units","sum"),
    defect_units=("defect_units","sum"), scheduled_hours=("scheduled_hours","sum"),
    operating_hours=("operating_hours","sum"))
summary["attainment_pct"] = 100*summary["produced_units"]/summary["planned_units"]
summary["defect_pct"] = 100*summary["defect_units"]/summary["produced_units"]
summary["units_per_hour"] = summary["produced_units"]/summary["operating_hours"]
summary["availability_pct"] = 100*summary["operating_hours"]/summary["scheduled_hours"]
display(summary.round(2))


## 3. 생산라인별 비교 차트
계획 달성률과 불량률을 각각 비교합니다.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].bar(summary["line"],summary["attainment_pct"])
axes[0].set(title="Production plan attainment by line",ylabel="Attainment (%)",xlabel="Line")
axes[1].bar(summary["line"],summary["defect_pct"])
axes[1].set(title="Defect rate by line",ylabel="Defect rate (%)",xlabel="Line")
for ax in axes: ax.grid(axis="y",alpha=.2)
plt.tight_layout(); plt.show()


## 4. 불량 유형 파레토 분석
불량 유형의 발생 건수를 내림차순 정렬하고 누적 비율을 계산합니다.

In [ ]:
pareto = defects.groupby("defect_type",as_index=False)["count"].sum().sort_values("count",ascending=False)
pareto["cumulative_pct"] = 100*pareto["count"].cumsum()/pareto["count"].sum()
display(pareto)
fig, ax = plt.subplots(figsize=(9,4))
x=np.arange(len(pareto))
ax.bar(x,pareto["count"]);ax.set_ylabel("Defect count")
ax.set_xticks(x, [f"Type {i+1}" for i in range(len(x))])
ax2=ax.twinx();ax2.plot(x,pareto["cumulative_pct"],marker="o")
ax2.set_ylim(0,110);ax2.set_ylabel("Cumulative (%)")
ax2.axhline(80,ls="--",color="gray",alpha=.6)
plt.title("Pareto of defect types (see table for Korean names)")
plt.tight_layout();plt.show()


## 5. B라인 전·후 비교
1~11일과 12~20일을 비교합니다. 차이가 관찰되더라도 **원인을 확정할 수 없습니다.**

In [ ]:
b = production[production["line"]=="B"].copy()
b["period"] = np.where(b["date"].dt.day <= 11,"days 1-11","days 12-20")
comparison=b.groupby("period",as_index=False).agg(produced=("produced_units","sum"),defects=("defect_units","sum"),planned=("planned_units","sum"))
comparison["defect_pct"]=100*comparison["defects"]/comparison["produced"]
comparison["attainment_pct"]=100*comparison["produced"]/comparison["planned"]
display(comparison.round(2))
b["daily_defect_pct"]=100*b["defect_units"]/b["produced_units"]
plt.figure(figsize=(9,4));plt.plot(b["date"],b["daily_defect_pct"],marker="o")
plt.axvline(pd.Timestamp("2026-10-12"),ls="--",color="gray")
plt.title("Line B daily defect rate");plt.ylabel("Defect rate (%)");plt.xticks(rotation=35);plt.grid(alpha=.2);plt.tight_layout();plt.show()


## 6. 생성형 AI 활용 및 결과 정리
`docs/PROMPTS.md`의 3~4번 프롬프트를 실행하세요. 위 `summary`, `pareto`, `comparison` 출력 표를 복사하여 Gemini에 붙여넣습니다. AI 답변을 원본 수치와 대조하고 `docs/WORKSHEET.md`를 완성합니다.

**추가 질문:** 파레토 차트에서 가장 많은 불량 유형이 왜 발생했는지, 현재 데이터만으로 알 수 있나요?